## DIAZOTROPS DATA
This notebook aims to explore the initial data sources for the environmental data

In [20]:
#we import all they key libraries needed in this notebook
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

## First dataset
This dataset is from the paper [Global oceanic diazotroph database version 2 and
elevated estimate of global oceanic N2 fixation](https://essd.copernicus.org/articles/15/3673/2023/essd-15-3673-2023-assets.html). It contains several datasets as sources. In particular there is a file dates 2023 and one dates 2024. It is written by the authors that the latter version has some erros fixed so I opted to us it here.

The first sheet I decided to examine as the xls file contains several is the cell count sheet. It clearly has coordinates that we can use to join it with environmental data, but the floating point values would not match perfectly with the NCEI dataset(there every number ends with 0.5). This can cause issues when joining as inner join can yield an empty result and the normal join based on indecies can create a lot of empty cells with each row containing only environmental or cell data but very rarely both. 

Thus, I am considering rounding the coordinates in both datasets in order to have higher chances of there being matches between coordinates.

In [21]:
dzdb_2024 = pd.read_csv("./csv/DiazotrophsDatabase-20240109.csv")
print(dzdb_2024.shape)
dzdb_2024.head()

(6765, 24)


,SOURCE: Related article,METHODS: Sampling/Analysis,DATE (yyyy-mm-dd),LATITUDE,LONGITUDE,DEPTH (m),Trichodesmium Colonies (x103 m-3),Trichodesmium Free trichomes (x106 m-3),Trichodesmium Total Trichomes (x106 m-3),UCYN-A Cells (x106 m-3),...,Richelia intracellularis (x106 m-3),Calothrix Associated Species,Calothrix intracellularis (x106 m-3),Temperature (˚C),Salinity (PSU),Nitrate (µM),Phosphate (µM),Fe (nM),Chlorophyll (mg m-3),Notes
0,"Borstad (1978) Thesis, McGill University, Canada",Standard Light Microscopy,7/9/1974,13.2,-59.7,0,NaN,NaN,0.67,NaN,...,NaN,NaN,NaN,27.5,32.67,NaN,NaN,NaN,0.12,NaN
1,NaN,Standard Light Microscopy,7/9/1974,13.2,-59.7,5,NaN,NaN,0.58,NaN,...,NaN,NaN,NaN,27.5,32.79,NaN,NaN,NaN,NaN,NaN
2,NaN,Standard Light Microscopy,7/9/1974,13.2,-59.7,10,NaN,NaN,0.69,NaN,...,NaN,NaN,NaN,27.5,32.54,NaN,NaN,NaN,0.20,NaN
3,NaN,Standard Light Microscopy,7/9/1974,13.2,-59.7,15,NaN,NaN,0.51,NaN,...,NaN,NaN,NaN,27.5,32.80,NaN,NaN,NaN,0.14,NaN
4,NaN,Standard Light Microscopy,7/9/1974,13.2,-59.7,25,NaN,NaN,0.43,NaN,...,NaN,NaN,NaN,27.0,33.48,NaN,NaN,NaN,0.19,NaN


## Null values of the dataset
When opening the dataset I noticed the abundance of empty cells I decided that one of the first check we need to perform is for empty values in columns that interest us the most. Clearly the best chances of producing the valid prediction we will have with "Trichodesmium Total Trichomes (x106 m-3) ", and the other 2 Trichodesmium column. 

However, the very low count of non empty cells for UCYN bacteria makes me doubtful if this will be enough.

In [22]:
print(dzdb_2024.notna().sum())
dzdb_2024.describe()

SOURCE: Related article                       84
METHODS: Sampling/Analysis                  6765
DATE (yyyy-mm-dd)                           6765
LATITUDE                                    6765
LONGITUDE                                   6765
DEPTH (m)                                   6765
Trichodesmium Colonies (x103 m-3)           1225
Trichodesmium Free trichomes (x106 m-3)     1541
Trichodesmium Total Trichomes (x106 m-3)    5681
UCYN-A Cells (x106 m-3)                       84
UCYN-B Cells (x106 m-3)                        4
UCYN-C Cells (x106 m-3)                        0
UCYN Cells (x106 m-3)                         81
Richelia Associated Species                 1704
Richelia intracellularis (x106 m-3)         2748
Calothrix Associated Species                  52
Calothrix intracellularis  (x106 m-3)         87
Temperature (˚C)                            1360
Salinity (PSU)                              1589
Nitrate (µM)                                 774
Phosphate (µM)      

,LATITUDE,LONGITUDE,Trichodesmium Free trichomes (x106 m-3),Trichodesmium Total Trichomes (x106 m-3),UCYN-B Cells (x106 m-3),UCYN-C Cells (x106 m-3),UCYN Cells (x106 m-3),Richelia intracellularis (x106 m-3),Calothrix intracellularis (x106 m-3),Temperature (˚C),Salinity (PSU),Fe (nM)
count,6765.00000,6765.000000,1541.000000,5681.000000,4.000000,0.0,81.000000,2748.000000,87.000000,1360.000000,1589.000000,19.000000
mean,11.89643,13.995217,9.586801,22.389078,92.556000,NaN,29.146432,3.170271,0.048326,23.582963,35.266281,0.269861
std,20.70263,103.349535,156.331984,688.149460,118.101372,NaN,94.227838,23.973336,0.268968,4.877795,1.659966,0.425774
min,-64.34000,-180.000000,0.000000,0.000000,0.224000,NaN,0.400000,0.000000,0.000000,10.310000,-9.000000,0.000009
25%,0.00000,-59.700000,0.000000,0.000000,0.760250,NaN,2.939000,0.000000,0.000000,20.512500,34.650000,0.000214
50%,13.25000,-19.980000,0.000000,0.000000,61.099500,NaN,5.540000,0.000000,0.000000,24.405000,35.300000,0.000750
75%,29.36000,123.070000,0.045000,0.070000,152.895250,NaN,26.100000,0.014825,0.000750,26.982500,36.170000,0.420000
max,78.25000,179.610000,4700.000000,44200.000000,247.801000,NaN,819.000000,766.000000,1.903400,126.190000,38.580000,1.560000


The second sheet we can examine is the integral cell count sheet which I saved as a separate csv file. We perform the same exact operations with it to see the overall statistics.

In [23]:
dzdb_2024_int = pd.read_csv("./csv/DiazotrophsDatabase-20240109_cellcount_int.csv")
print(dzdb_2024_int.shape)
dzdb_2024_int.head()

(1411, 21)


,SOURCE: Related article,SOURCE: Related article.1,DATE (yyyy-mm-dd),LATITUDE,LONGITUDE,INTEGRAL DEPTH (m),Trichodesmium Colonies (x103 m-2),Trichodesmium Free trichomes (x106 m-2),Trichodesmium Total Trichomes (x106 m-2),UCYN Cells (x106 m-2),...,Richelia intracellularis (x106 m-2),Calothrix Associated Species,Calothrix intracellularis (x106 m-2),Surface Temperature (˚C),Surface Salinity (PSU),Surface Nitrate (µM),Surface Phosphate (µM),Surface Fe (nM),Chlorophyll (mg m-2),Notes
0,"Benavides et al. (2011), doi:10.3354/ame01534",Standard Light Microscopy,2009-07-26,41.5,-9.32,65.0,NaN,0.000,0.000,NaN,...,NaN,NaN,NaN,18.60,35.70,0.61,0.06,NaN,0.15,2-3 tows performed with 50 µm net
1,NaN,Standard Light Microscopy,2009-07-27,41.5,-10.58,60.0,NaN,0.002,0.002,NaN,...,NaN,NaN,NaN,19.75,35.87,0.13,0.02,NaN,NaN,2-3 tows performed with 50 µm net
2,NaN,Standard Light Microscopy,2009-07-28,41.5,-12.26,62.0,NaN,0.003,0.003,NaN,...,NaN,NaN,NaN,19.81,35.99,0.05,0.02,NaN,0.02,2-3 tows performed with 50 µm net
3,NaN,Standard Light Microscopy,2009-07-30,41.5,-15.29,67.0,NaN,0.001,0.001,NaN,...,NaN,NaN,NaN,19.75,35.96,0.09,0.02,NaN,NaN,2-3 tows performed with 50 µm net
4,NaN,Standard Light Microscopy,2009-07-31,41.5,-17.18,53.0,NaN,0.007,0.007,NaN,...,NaN,NaN,NaN,18.95,35.87,0.05,0.03,NaN,0.40,2-3 tows performed with 50 µm net


In [24]:
print(dzdb_2024_int.notna().sum())
dzdb_2024_int.describe()

SOURCE: Related article                       47
SOURCE: Related article.1                   1411
DATE (yyyy-mm-dd)                           1411
LATITUDE                                    1411
LONGITUDE                                   1411
INTEGRAL DEPTH (m)                          1411
Trichodesmium Colonies (x103 m-2)            393
Trichodesmium Free trichomes (x106 m-2)      522
Trichodesmium Total Trichomes (x106 m-2)    1297
UCYN Cells (x106 m-2)                         19
Richelia Associated Species                  406
Richelia intracellularis (x106 m-2)          435
Calothrix Associated Species                   8
Calothrix intracellularis  (x106 m-2)         17
Surface Temperature (˚C)                     459
Surface Salinity (PSU)                       469
Surface Nitrate (µM)                         228
Surface Phosphate (µM)                       276
Surface Fe (nM)                              120
Chlorophyll  (mg m-2)                        173
Notes               

,LATITUDE,LONGITUDE,INTEGRAL DEPTH (m),Trichodesmium Colonies (x103 m-2),Trichodesmium Free trichomes (x106 m-2),Trichodesmium Total Trichomes (x106 m-2),UCYN Cells (x106 m-2),Richelia intracellularis (x106 m-2),Calothrix intracellularis (x106 m-2),Surface Temperature (˚C),Surface Salinity (PSU),Surface Nitrate (µM),Surface Phosphate (µM),Surface Fe (nM),Chlorophyll (mg m-2)
count,1411.000000,1411.000000,1411.000000,393.000000,522.000000,1297.000000,19.000000,435.000000,17.000000,459.000000,469.000000,228.000000,276.000000,120.000000,173.000000
mean,16.946736,9.103940,109.081148,28.113821,10.608421,10.391441,1687.775158,538.423777,19.245118,26.643268,34.611748,0.128114,0.394493,1.612167,13.301908
std,16.696257,106.124185,71.222798,86.381629,45.076470,34.917039,2035.345391,2646.232524,41.391709,5.222897,2.121061,0.330944,0.780058,1.747653,8.197072
min,-40.500000,-180.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,15.080000,22.770000,0.000000,0.000000,0.090000,0.020000
25%,11.020000,-59.700000,57.000000,0.000000,0.058350,0.035400,595.629000,0.112500,0.000000,25.960000,33.960000,0.020000,0.020000,0.580000,5.880000
50%,21.571203,-30.740000,98.000000,0.000000,0.572150,0.600000,1015.000000,0.998000,0.000000,27.050000,35.200000,0.050000,0.050000,1.220000,12.700000
75%,28.500000,122.600000,175.000000,7.629000,2.954350,5.700000,1781.500000,4.979889,8.678000,27.900000,36.000000,0.065000,0.245000,2.170000,19.610000
max,41.840000,201.170000,1250.000000,586.683000,660.000000,660.000000,8208.600000,26039.992000,144.445000,126.190000,37.600000,3.010000,4.240000,11.760000,35.110000
